Q-learn
Principal code                         

Camilo Vásquez

In [9]:
import random

SIZE = 4
START = (0, 0)
GOAL = (3, 3)
BASE_WALLS = {(1, 1), (2, 1)}
ACTIONS = [(0, 1), (1, 0), (0, -1), (-1, 0)]
ACTION_NAMES = ["right", "down", "left", "up"]

def open_states(walls):
    return [(r, c) for r in range(SIZE) for c in range(SIZE) if (r, c) not in walls]

def step_env(state, action, walls, step_reward=-1):
    row, col = state
    proposed = (max(0, min(SIZE - 1, row + action[0])), max(0, min(SIZE - 1, col + action[1])))
    next_state = state if proposed in walls else proposed
    done = next_state == GOAL
    return next_state, (10 if done else step_reward), done

def train_q(epsilon, step_reward=-1, walls=BASE_WALLS, alpha=0.2, gamma=0.95,
            episodes=1000, max_steps=40, seed=7):
    random.seed(seed)
    q = {s: [0.0] * len(ACTIONS) for s in open_states(walls)}

    def choose(state):
        if random.random() < epsilon:
            return random.randrange(len(ACTIONS))
        best = 0
        for i in range(1, len(ACTIONS)):
            if q[state][i] > q[state][best]:
                best = i
        return best

    for _ in range(episodes):
        state = START
        for _ in range(max_steps):
            a = choose(state)
            next_state, reward, done = step_env(state, ACTIONS[a], walls, step_reward)
            target = reward if done else reward + gamma * max(q[next_state])
            q[state][a] += alpha * (target - q[state][a])
            state = next_state
            if done:
                break
    return q

def greedy_route(q, walls):
    state, route = START, [START]
    for _ in range(SIZE * SIZE * 2):
        if state == GOAL:
            break
        a = q[state].index(max(q[state]))
        state, _, done = step_env(state, ACTIONS[a], walls)
        route.append(state)
        if done:
            break
    return route

def report(label, q, walls, show_route=False):
    route = greedy_route(q, walls)
    start_q = q[START]
    print(label)
    print("  Q at START (right, down, left, up):", [round(v, 3) for v in start_q])
    print("  Greedy action:", ACTION_NAMES[start_q.index(max(start_q))])
    print("  Route length:", len(route) - 1, "| Goal reached:", route[-1] == GOAL)
    if show_route:
        print("  Route:", route)

1. Compare $\epsilon=0.0$, $0.2$, and $0.5$. Record the Q values at START, greedy action, route length, and goal result. In this grid, even $\epsilon=0$ reaches the goal: explain how the first-action tie-breaking and zero-valued untried actions affect the sequence of choices. Why does this not guarantee reliable discovery in every environment?

In [10]:
q = train_q(epsilon=0.0)
report("epsilon = 0.0", q, BASE_WALLS)

epsilon = 0.0
  Q at START (right, down, left, up): [3.213, -2.525, -2.625, -2.625]
  Greedy action: right
  Route length: 6 | Goal reached: True


In [4]:
q = train_q(epsilon=0.2)
report("epsilon = 0.2", q, BASE_WALLS)

epsilon = 0.2
  Q at START (right, down, left, up): [3.213, 3.202, 2.053, 2.053]
  Greedy action: right
  Route length: 6 | Goal reached: True


In [5]:
q = train_q(epsilon=0.5)
report("epsilon = 0.5", q, BASE_WALLS)

epsilon = 0.5
  Q at START (right, down, left, up): [3.213, 3.213, 2.053, 2.053]
  Greedy action: right
  Route length: 6 | Goal reached: True


Control: same test with step_reward = 0.

In [6]:
for eps in (0.0, 0.2):
    q = train_q(epsilon=eps, step_reward=0)
    report(f"step_reward = 0, epsilon = {eps}", q, BASE_WALLS)

step_reward = 0, epsilon = 0.0
  Q at START (right, down, left, up): [0.0, 0.0, 0.0, 0.0]
  Greedy action: right
  Route length: 32 | Goal reached: False
step_reward = 0, epsilon = 0.2
  Q at START (right, down, left, up): [7.738, 6.982, 7.351, 7.351]
  Greedy action: right
  Route length: 6 | Goal reached: True


Explanation: All three runs get to the goal with the same 6 move route. Initially , all Q values are 0 , so the tie breaks in favor of the first action , right . Its reward is negative, so its Q dips below 0 and the untried actions, still at 0, look relatively better. The agent tries each action in turn. It finds the goal even with epsilon 0. This is not reliable: it works because the step rewards are negative only. If step_reward is 0, Q is 0, ties always go right, and the goal is never found.

2. With $\epsilon=0.2$ held fixed, change `step_reward` in the Section 4 cell from $-1$ to $-2$. Compare the Q values and route. Does the route change here? Explain why a larger step cost lowers returns, and what kind of environment would make the shorter-route preference visible.

In [11]:
q = train_q(epsilon=0.2, step_reward=-2)
report("step_reward = -2", q, BASE_WALLS, show_route=True)

step_reward = -2
  Q at START (right, down, left, up): [-1.311, -1.363, -3.245, -3.245]
  Greedy action: right
  Route length: 6 | Goal reached: True
  Route: [(0, 0), (0, 1), (0, 2), (1, 2), (1, 3), (2, 3), (3, 3)]


Explanation: Q at START drops about -1.3 rather than 3.2 because each step costs more. The route changes cells but is still 6 moves. Several routes have the same true return and training noise breaks the tie. The preference for shorter routes is only apparent when routes of different lengths compete, for example a near goal with a small reward vs a far goal with a large reward.

3. Add a wall at `(1, 2)` while keeping a path to the goal, then rerun training and test the learned route.

In [8]:
walls_extra = BASE_WALLS | {(1, 2)}
q = train_q(epsilon=0.2, walls=walls_extra)
report("extra wall at (1, 2)", q, walls_extra, show_route=True)

extra wall at (1, 2)
  Q at START (right, down, left, up): [3.213, 3.051, 2.053, 2.053]
  Greedy action: right
  Route length: 6 | Goal reached: True
  Route: [(0, 0), (0, 1), (0, 2), (0, 3), (1, 3), (2, 3), (3, 3)]
